# 5. Framing, features, baselines, and the dispatch harness

This is the methodological backbone. We define the day-ahead
forecasting task, build a leakage-safe backtest, establish baselines,
and — critically — introduce the **dispatch harness** so every model
reports its economic value via capture ratio.

## Objectives

- Define the day-ahead forecasting task: at 12:00 today, forecast the next 48 half-hours.
- Build the feature matrix with lags, calendar, and renewable forecasts.
- Implement and demonstrate the rolling-origin backtest with embargo.
- Score similar-day naive and autoregressive baselines.
- Implement the battery arbitrage LP (`dispatch.schedule`).
- Build the rolling MPC (`dispatch.rolling_mpc`).
- Load AEMO pre-dispatch as the operator benchmark.
- Report the full scorecard: MAE, rMAE, CRPS, and capture ratio.

## Prerequisites

- Notebooks 01–04 completed (parquet datasets cached).
- `nemseer` installed (`pip install nemseer`) for AEMO pre-dispatch.
- `cvxpy` installed for the dispatch LP.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from grian.backtest import rolling_origin
from grian.config import load_config
from grian.dispatch import capture_ratio, rolling_mpc, schedule
from grian.features import build_matrix
from grian.metrics import mae, relative_mae
from grian.models.baselines import autoregression, similar_day_naive
from grian.viz import apply_style, save_fig

cfg = load_config()
apply_style()
warnings.filterwarnings("ignore", category=FutureWarning)

REGION = cfg["region"]
TRAIN_START = cfg["train_start"]
TRAIN_END = cfg["train_end"]
TEST_START = cfg["test_start"]
TEST_END = cfg["test_end"]
HORIZON = cfg["horizon_periods"]
SEED = cfg["seed"]

battery = cfg["battery"]
print(f"Region: {REGION}")
print(f"Train: {TRAIN_START} to {TRAIN_END}")
print(f"Test:  {TEST_START} to {TEST_END}")
print(f"Horizon: {HORIZON} periods (= {HORIZON * 30 / 60:.0f} hours)")
print(f"Battery: {battery['power_mw']} MW / {battery['duration_hours']}h / "
      f"{battery['efficiency_roundtrip']*100:.0f}% RT eff / "
      f"{battery['max_cycles_per_day']} cycles/day")

---
## 1. Day-ahead framing

At **12:00 today**, forecast the price for each of the next **48
half-hours** (12:00 today through 12:00 tomorrow). This matches the
AEMO pre-dispatch timeline.

What's known at forecast time:
- All prices up to 11:30 today (lag-1 is the most recent).
- All demand up to 11:30.
- Weather forecasts for the next 48 hours.
- Calendar (hour, day-of-week, month) for the forecast window.

What's NOT known:
- Actual prices during the forecast window (obviously).
- Actual demand during the forecast window.
- Actual renewable generation during the forecast window.

The **embargo** in the backtest ensures that the training data stops
before the forecast window begins — specifically, we drop `horizon`
periods between the end of training and the start of the forecast.

In [ ]:
# Load the 30-min dataset
processed = Path(cfg["paths"]["processed"])
df = pd.read_parquet(processed / f"{REGION}_30min.parquet")

print(f"Dataset: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Period: {df.index[0]} to {df.index[-1]}")
df.head()

---
## 2. Feature matrix

Build the feature matrix in-cell first to see what each feature
looks like, then verify that `features.build_matrix()` produces
the same result.

In [ ]:
# Build features manually first
manual = df[["price"]].copy()

# Lagged prices: same time yesterday (48), 2 days ago (96), 1 week ago (336)
for lag in [48, 96, 336]:
    manual[f"price_lag_{lag}"] = manual["price"].shift(lag)

# Calendar features
manual["hour"] = manual.index.hour
manual["day_of_week"] = manual.index.dayofweek
manual["month"] = manual.index.month

# Demand (lagged by 1 to avoid leakage — we know demand up to lag-1)
if "demand" in df.columns:
    manual["demand_lag_1"] = df["demand"].shift(1)
    manual["demand_lag_48"] = df["demand"].shift(48)

manual = manual.dropna()
print(f"Manual feature matrix: {manual.shape}")
print(f"Features: {list(manual.columns)}")
manual.head()

In [ ]:
# Now use the library function
feature_matrix = build_matrix(df[["price"]], df[["demand"]] if "demand" in df.columns else None)
print(f"Library feature matrix: {feature_matrix.shape}")
print(f"Features: {list(feature_matrix.columns)}")
feature_matrix.head()

Each feature group serves a purpose:

- **Price lags** — autocorrelation (lag-48 is the strongest signal).
- **Calendar** — diurnal, weekly, seasonal patterns.
- **Demand lags** — recent demand level predicts upcoming prices.

All features are strictly available at forecast time — no future
leakage.

---
## 3. Rolling-origin backtest

The backtest walks forward through the test period:

1. Train on all data up to the **origin**.
2. Skip an **embargo** gap (= horizon, to prevent leakage).
3. Forecast the next **horizon** periods.
4. Compare to actual prices.
5. Advance by **step** periods and repeat.

Why the embargo? Without it, the most recent training observation
is only 1 period before the first forecast target — and with lag-1
features, that's a direct leak.

In [ ]:
# Demonstrate on the similar-day naive baseline
def naive_model_fn(train_data, horizon):
    """Wrap similar_day_naive for the backtest interface."""
    origin = train_data.index[-1]
    return similar_day_naive(train_data[["price"]], origin, horizon)

results_naive = rolling_origin(
    data=df,
    model_fn=naive_model_fn,
    train_start=TRAIN_START,
    test_start=TEST_START,
    test_end=TEST_END,
    horizon=HORIZON,
    step=HORIZON,
)

print(f"Backtest windows: {len(results_naive)}")
print(f"First origin: {results_naive[0]['origin']}")
print(f"Last origin:  {results_naive[-1]['origin']}")

In [ ]:
# Score the naive baseline
naive_actuals = np.concatenate([r["actual"] for r in results_naive])
naive_forecasts = np.concatenate([r["forecast"] for r in results_naive])

naive_mae = mae(naive_actuals, naive_forecasts)
print(f"Similar-day naive MAE: ${naive_mae:.2f}/MWh")

In [ ]:
# Plot a few sample forecast windows
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, r in zip(axes.flat, results_naive[:4]):
    idx = pd.date_range(r["origin"] + pd.Timedelta(minutes=30),
                        periods=len(r["actual"]), freq="30min")
    ax.plot(idx, r["actual"], "k-", linewidth=1, label="Actual")
    ax.plot(idx, r["forecast"], "--", linewidth=1, label="Naive forecast")
    ax.set_title(f"Origin: {r['origin'].strftime('%Y-%m-%d %H:%M')}")
    ax.set_ylabel("$/MWh")
    ax.legend(fontsize=8)

fig.suptitle(f"{REGION} — similar-day naive forecast samples", fontsize=13)
fig.tight_layout()
save_fig(fig, "05_naive_forecast_samples")
plt.show()

---
## 4. Autoregressive baseline

In [ ]:
def ar_model_fn(train_data, horizon):
    """Wrap autoregression for the backtest interface."""
    origin = train_data.index[-1]
    return autoregression(train_data[["price"]], origin, horizon)

results_ar = rolling_origin(
    data=df,
    model_fn=ar_model_fn,
    train_start=TRAIN_START,
    test_start=TEST_START,
    test_end=TEST_END,
    horizon=HORIZON,
    step=HORIZON,
)

ar_actuals = np.concatenate([r["actual"] for r in results_ar])
ar_forecasts = np.concatenate([r["forecast"] for r in results_ar])
ar_mae_val = mae(ar_actuals, ar_forecasts)
ar_rmae = relative_mae(ar_actuals, ar_forecasts, naive_forecasts)

print(f"AR baseline MAE: ${ar_mae_val:.2f}/MWh")
print(f"AR relative MAE (vs naive): {ar_rmae:.3f}")

---
## 5. The dispatch harness

This is the key section. A price forecast is only as good as the
**money it makes**. We formalise this via the battery arbitrage LP.

### 5a. The maths

Maximise revenue = $\sum_t p_t \times (d_t - c_t) \times \Delta t$

Subject to:
- $0 \leq c_t \leq P_{max}$ (charge power limit)
- $0 \leq d_t \leq P_{max}$ (discharge power limit)
- $0 \leq E_t \leq E_{max}$ (state of charge limits)
- $E_{t+1} = E_t + c_t \sqrt{\eta} \Delta t - d_t / \sqrt{\eta} \Delta t$ (energy balance)
- Total throughput $\leq$ max_cycles $\times E_{max}$ (cycle limit)

### 5b. Perfect foresight — the upper bound

In [ ]:
# Perfect foresight on one test day
test_day = df.loc[TEST_START].iloc[:HORIZON]
day_prices = test_day["price"].values

result = schedule(
    day_prices,
    power_mw=battery["power_mw"],
    duration_hours=battery["duration_hours"],
    efficiency=battery["efficiency_roundtrip"],
    max_cycles=battery["max_cycles_per_day"],
)

print(f"Status: {result['status']}")
print(f"Revenue: ${result['revenue']:,.0f}")

In [ ]:
# Plot charge/discharge vs price for one day
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

hours = np.arange(HORIZON) * 0.5
ax1.plot(hours, day_prices, "k-", linewidth=1.5, label="Price")
ax1.set_ylabel("Price ($/MWh)")
ax1.set_title(f"Perfect-foresight dispatch — {TEST_START}")
ax1.legend()

net_power = np.array(result["discharge"]) - np.array(result["charge"])
colors = ["C2" if p > 0 else "C3" for p in net_power]
ax2.bar(hours, net_power, width=0.4, color=colors, alpha=0.7)
ax2_twin = ax2.twinx()
ax2_twin.plot(hours, result["soc"], "b-", linewidth=1.5, label="SoC")
ax2.set_xlabel("Hour")
ax2.set_ylabel("Net power (MW): green=discharge, red=charge")
ax2_twin.set_ylabel("State of charge (MWh)")
ax2_twin.legend(loc="upper right")

fig.tight_layout()
save_fig(fig, "05_perfect_foresight_dispatch")
plt.show()

The battery charges during low-price periods (midday solar surplus)
and discharges during high-price periods (evening peak). This is the
perfect-foresight upper bound — no forecast can do better.

### 5c. Naive MPC — point-forecast dispatch

At each half-hour, take the point forecast, solve the LP, execute
one period, then re-forecast. This is the simplest way to translate
a forecast into dispatch decisions.

In [ ]:
# Run naive MPC with the similar-day naive forecast over one test week
test_week_prices = df.loc[TEST_START:pd.Timestamp(TEST_START) + pd.Timedelta(days=7)]["price"].values

def naive_forecaster(t, horizon):
    """Return the similar-day naive forecast from position t."""
    # Use same-time-yesterday as forecast
    if t >= 48:
        return test_week_prices[max(0, t-48):max(0, t-48)+horizon]
    return np.full(horizon, np.median(test_week_prices[:max(1, t)]))

mpc_result = rolling_mpc(
    test_week_prices,
    forecaster=naive_forecaster,
    horizon=HORIZON,
    power_mw=battery["power_mw"],
    duration_hours=battery["duration_hours"],
    efficiency=battery["efficiency_roundtrip"],
    max_cycles=battery["max_cycles_per_day"],
)

# Perfect foresight for the same period (daily chunks)
perfect_rev = 0
n_days = len(test_week_prices) // HORIZON
for d in range(n_days):
    day_p = test_week_prices[d * HORIZON:(d + 1) * HORIZON]
    if len(day_p) == HORIZON:
        r = schedule(day_p, **{k: battery[k] for k in
                     ["power_mw", "duration_hours", "efficiency_roundtrip", "max_cycles_per_day"]}
                     if False else schedule(day_p, battery["power_mw"],
                     battery["duration_hours"], battery["efficiency_roundtrip"],
                     battery["max_cycles_per_day"]))
        perfect_rev += r["revenue"]

cr = capture_ratio(mpc_result["total_revenue"], perfect_rev)
print(f"Naive MPC revenue:     ${mpc_result['total_revenue']:,.0f}")
print(f"Perfect foresight:     ${perfect_rev:,.0f}")
print(f"Capture ratio:         {cr:.1%}")

---
## 6. AEMO pre-dispatch benchmark

AEMO publishes pre-dispatch price forecasts via NEMSEER. This is
what a battery operator would see — the official market forecast.
Our models need to beat this.

In [ ]:
# Try loading NEMSEER pre-dispatch
try:
    from nemseer import compile_data, download_raw_data

    download_raw_data(
        "PREDISPATCH",
        "PRICE",
        TEST_START,
        pd.Timestamp(TEST_START) + pd.Timedelta(days=30),
        Path(cfg["nemseer_cache"]),
    )
    predispatch = compile_data(
        "PREDISPATCH",
        "PRICE",
        TEST_START,
        pd.Timestamp(TEST_START) + pd.Timedelta(days=30),
        Path(cfg["nemseer_cache"]),
        filter_cols={"REGIONID": [REGION]},
    )
    has_nemseer = True
    print(f"NEMSEER pre-dispatch loaded: {predispatch.shape}")
except Exception as e:
    has_nemseer = False
    print(f"NEMSEER not available: {e}")
    print("Proceeding without AEMO pre-dispatch benchmark.")
    print("Install nemseer: pip install nemseer")

---
## 7. Full scorecard

Score all baselines end-to-end: backtest → forecast → MPC → capture
ratio.

In [ ]:
# Compute perfect-foresight revenue over the test period
test_prices = df.loc[TEST_START:TEST_END]["price"].dropna().values
n_test_days = len(test_prices) // HORIZON

perfect_total = 0
for d in range(n_test_days):
    day_p = test_prices[d * HORIZON:(d + 1) * HORIZON]
    if len(day_p) == HORIZON:
        r = schedule(day_p, battery["power_mw"], battery["duration_hours"],
                     battery["efficiency_roundtrip"], battery["max_cycles_per_day"])
        perfect_total += r["revenue"]

print(f"Perfect-foresight revenue over test period: ${perfect_total:,.0f}")
print(f"Test days: {n_test_days}")

In [ ]:
# Build scorecard
scorecard = []

for name, results in [("Similar-day naive", results_naive), ("AR", results_ar)]:
    actuals = np.concatenate([r["actual"] for r in results])
    forecasts = np.concatenate([r["forecast"] for r in results])

    model_mae = mae(actuals, forecasts)
    model_rmae = relative_mae(actuals, forecasts, naive_forecasts)

    # MPC revenue (approximate: use daily chunks of forecasts)
    model_rev = 0
    for r in results:
        fc = np.array(r["forecast"])
        if len(fc) == HORIZON:
            sched = schedule(fc, battery["power_mw"], battery["duration_hours"],
                             battery["efficiency_roundtrip"], battery["max_cycles_per_day"])
            # Revenue is earned at actual prices with the schedule from forecast
            act = np.array(r["actual"])
            charge = np.array(sched["charge"])
            discharge = np.array(sched["discharge"])
            model_rev += np.sum(act * (discharge - charge) * 0.5)

    cr = capture_ratio(model_rev, perfect_total) if perfect_total > 0 else 0

    scorecard.append({
        "Model": name,
        "MAE ($/MWh)": model_mae,
        "rMAE vs naive": model_rmae,
        "Capture ratio": cr,
    })

scorecard_df = pd.DataFrame(scorecard).set_index("Model")
scorecard_df.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
scorecard_df["Capture ratio"].plot(kind="barh", ax=ax, color="C0", alpha=0.7)
ax.set_xlabel("Capture ratio")
ax.set_title("Baseline capture ratios")
ax.axvline(1.0, color="red", linewidth=0.8, linestyle="--", label="Perfect foresight")
ax.legend()
fig.tight_layout()
save_fig(fig, "05_baseline_capture_ratios")
plt.show()

---
## Exercises

### Exercise 1: The embargo — prove it prevents leakage

Remove the embargo (set it to 0), rerun the naive backtest, and show
that the score improves. Why does this happen?

<details><summary>Hint 1</summary>

Call `rolling_origin` with `embargo=0`. Compare the MAE to the
version with the default embargo.

</details>

<details><summary>Hint 2</summary>

Without the embargo, the training data includes observations that
are only 1 period before the forecast target. With lag-1 features,
the model effectively sees the answer.

</details>

<details><summary>Hint 3</summary>

The improvement is artificial — it wouldn't exist in production
because you'd never have the data that close to the forecast target
at forecast time.

</details>

<details><summary>Solution</summary>

```python
results_no_embargo = rolling_origin(
    data=df,
    model_fn=naive_model_fn,
    train_start=TRAIN_START,
    test_start=TEST_START,
    test_end=TEST_END,
    horizon=HORIZON,
    step=HORIZON,
    embargo=0,
)

no_emb_actuals = np.concatenate([r["actual"] for r in results_no_embargo])
no_emb_forecasts = np.concatenate([r["forecast"] for r in results_no_embargo])
no_emb_mae = mae(no_emb_actuals, no_emb_forecasts)

print(f"MAE with embargo:    ${naive_mae:.2f}/MWh")
print(f"MAE without embargo: ${no_emb_mae:.2f}/MWh")
print(f"Improvement: {(naive_mae - no_emb_mae) / naive_mae:.1%} (artificial!)")
```

The score improves because without the embargo, the model can see
prices very close to the forecast window. In production, this data
wouldn't be available at forecast time, so the improvement is
leakage, not skill.

</details>

In [ ]:
# Your analysis here

### Exercise 2: Battery sizing sensitivity

Change `duration_hours` from 2 to 4. How does perfect-foresight
revenue change? Does the relative ranking of models change?

<details><summary>Hint 1</summary>

Rerun `schedule()` on one day with the new battery parameters.
Compare revenue.

</details>

<details><summary>Hint 2</summary>

A 4-hour battery can store more energy, so it can capture larger
price swings. But it also needs longer to charge, so it may miss
short spikes.

</details>

<details><summary>Hint 3</summary>

Plot revenue as a function of duration_hours (1, 2, 4, 8). Does
the marginal value of storage flatten out?

</details>

<details><summary>Solution</summary>

```python
durations = [1, 2, 4, 6, 8]
revenues = []

for dur in durations:
    r = schedule(day_prices, battery["power_mw"], dur,
                 battery["efficiency_roundtrip"], battery["max_cycles_per_day"])
    revenues.append(r["revenue"])
    print(f"  {dur}h: ${r['revenue']:,.0f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(durations, revenues, "o-", markersize=8)
ax.set_xlabel("Battery duration (hours)")
ax.set_ylabel("Perfect-foresight revenue ($)")
ax.set_title("Revenue vs battery duration (one day)")
fig.tight_layout()
plt.show()
```

Revenue increases with duration but with diminishing returns. A 4h
battery captures most of the daily price spread; going to 8h adds
little because there aren't enough distinct charge/discharge cycles
in a day to use the extra capacity.

</details>

In [ ]:
# Your analysis here

### Exercise 3: When does a bad forecast make money?

Is capture ratio monotonically related to forecast MAE? Can a worse
forecast sometimes produce a higher capture ratio?

<details><summary>Hint 1</summary>

Think about correlation between forecast errors and price level.
A forecast that overestimates evening prices will dispatch more
aggressively during the peak — if the peak still materialises
(just lower than forecast), it still captures revenue.

</details>

<details><summary>Hint 2</summary>

Create an artificially noisy forecast by adding random noise to
the naive forecast. Try different noise levels and plot MAE vs
capture ratio.

</details>

<details><summary>Hint 3</summary>

The relationship is generally negative (better forecast → more
revenue), but it's not perfectly monotonic. The dispatch LP is
sensitive to the *ranking* of prices within a day, not just
their absolute values.

</details>

<details><summary>Solution</summary>

```python
rng = np.random.default_rng(SEED)
noise_levels = [0, 10, 25, 50, 100, 200]
mae_cr_pairs = []

for noise in noise_levels:
    total_rev = 0
    all_mae = []
    for r in results_naive:
        fc = np.array(r["forecast"]) + rng.normal(0, noise, len(r["forecast"]))
        act = np.array(r["actual"])
        all_mae.append(mae(act, fc))
        if len(fc) == HORIZON:
            sched = schedule(fc, battery["power_mw"], battery["duration_hours"],
                             battery["efficiency_roundtrip"], battery["max_cycles_per_day"])
            total_rev += np.sum(act * (np.array(sched["discharge"]) - np.array(sched["charge"])) * 0.5)
    cr = capture_ratio(total_rev, perfect_total)
    avg_mae = np.mean(all_mae)
    mae_cr_pairs.append((noise, avg_mae, cr))
    print(f"  noise σ={noise:>3}: MAE=${avg_mae:.1f}, CR={cr:.3f}")

fig, ax = plt.subplots(figsize=(8, 5))
maes = [p[1] for p in mae_cr_pairs]
crs = [p[2] for p in mae_cr_pairs]
ax.plot(maes, crs, "o-", markersize=8)
ax.set_xlabel("MAE ($/MWh)")
ax.set_ylabel("Capture ratio")
ax.set_title("Forecast quality vs economic value")
for p in mae_cr_pairs:
    ax.annotate(f"σ={p[0]}", (p[1], p[2]), textcoords="offset points",
                xytext=(5, 5), fontsize=8)
fig.tight_layout()
plt.show()
```

Generally, better MAE → higher capture ratio. But small amounts
of noise don't always hurt — the LP cares about price *ranking*
within a day, not absolute accuracy. This is why capture ratio,
not MAE, is the headline metric.

</details>

In [ ]:
# Your analysis here

---
## What we learned

1. The day-ahead framing: at 12:00, forecast 48 half-hours ahead.
2. The feature matrix uses lags, calendar, and demand — all strictly
   available at forecast time.
3. The rolling-origin backtest with embargo prevents leakage.
4. The similar-day naive baseline sets the floor; AR improves on it.
5. The dispatch LP converts a price forecast into a charge/discharge
   schedule that maximises revenue.
6. **Capture ratio** is the headline metric: model revenue as a
   fraction of perfect-foresight revenue.
7. Forecast quality and economic value are correlated but not
   perfectly — the LP cares about price ranking, not just MAE.

**Next:** Notebook 06 builds the LEAR model — the strong linear
benchmark that ML models must beat.

In [ ]:
# Write report
report_dir = Path(cfg["paths"]["reports"])
report_dir.mkdir(parents=True, exist_ok=True)

report = f"""# Notebook 05 — Framing, Features, Baselines Report

Region: {REGION}
Train: {TRAIN_START} to {TRAIN_END}
Test: {TEST_START} to {TEST_END}
Horizon: {HORIZON} periods

## Scorecard

{scorecard_df.to_markdown()}

## Perfect foresight

- Revenue: ${perfect_total:,.0f}
- Battery: {battery['power_mw']} MW / {battery['duration_hours']}h

## Implemented

- `features.build_matrix()` — feature matrix with lags, calendar, demand
- `backtest.rolling_origin()` — rolling-origin backtest with embargo
- `models.baselines.similar_day_naive()` — same day-of-week last week
- `models.baselines.autoregression()` — simple AR baseline
- `dispatch.schedule()` — battery arbitrage LP (cvxpy)
- `dispatch.rolling_mpc()` — rolling model-predictive control
"""

(report_dir / "05_framing_features_baselines.md").write_text(report)
print("Report written to", report_dir / "05_framing_features_baselines.md")